In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from keras.models import Sequential
from keras.layers import Dense, LSTM, Embedding
from keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# External dataset is intentionally omitted from the repository.
DATA_PATH = "path/to/sentiment-emotion-labelled_Dell_tweets.csv"
df = pd.read_csv(DATA_PATH)


In [ ]:
df = pd.read_csv("path/to/external/data")

## **1.?곗씠??遺꾩꽍?섍린**

### **1-1. ?곗씠???뺤씤 諛??꾩쿂由?*

In [ ]:
df

Datatime : ?몄쐵 ?묒꽦???좎쭨&?쒓컙

Tweet ID : ?몄쐵 ?묒꽦???꾩씠??n
Text : ?몄쐵 ?댁슜

Username : ?몄쐞???ъ슜???대쫫

sentiment : 媛먯젙 (湲띿젙/以묐┰/遺??

sentiment_score : 媛먯젙 ?먯닔(媛먯젙 媛뺣룄瑜??섏튂濡??섑???

emotion : 媛먯젙 ?좏삎

emotion_score : 媛먯젙 ?좏삎 ?먯닔

In [ ]:
df = df.sample(n=1000, random_state=42)

### **1-2. ?몄쐵 ?묒꽦???꾩씠??諛??ъ슜??遺꾩꽍**

In [ ]:
tweet_id_counts = df['Tweet Id'].value_counts()

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
tweet_id_counts.plot(kind='bar', color='blue')

plt.title('Tweet Id Distribution')
plt.xlabel('Tweet Id')
plt.ylabel('Frequency')
plt.xticks([])

username_counts = df['Username'].value_counts()
plt.subplot(1, 2, 2)
username_counts.plot(kind='bar', color='red')
plt.title('Usernames Distribution')
plt.xlabel('Username')
plt.ylabel('Frequency')

plt.tight_layout()
plt.xticks([])
plt.show()

**?쒖そ?쇰줈 ?좊┛ ?곗씠?캸 -> ?섎??놁쓬 -> ?대떦 ????젣**

In [ ]:
df.drop(columns=['Unnamed: 0','Tweet Id', 'Username'], inplace=True)
df

### **1-3. sentiment , emotion ?먭렇?섑봽濡??쒓컖??*

In [ ]:
sentiment_counts = df['sentiment'].value_counts()

plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.pie(sentiment_counts, labels=sentiment_counts.index, autopct='%1.1f%%', startangle=140)
plt.title('Sentiment Distribution')

emotion_counts = df['emotion'].value_counts()

plt.subplot(1, 2, 2)
plt.pie(emotion_counts, labels=emotion_counts.index, autopct='%1.1f%%', startangle=140)
plt.title('Emotion Distribution')

plt.tight_layout()
plt.show()

### **1-4. emotion 留됯렇?섑봽濡??쒓컖??*

In [ ]:
emotion_counts = df['emotion'].value_counts()

plt.figure(figsize=(10, 6))

bars = plt.bar(emotion_counts.index, emotion_counts.values)

plt.title('Emotion Distribution')
plt.xlabel('Emotion')
plt.ylabel('Frequency')

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval, int(yval), ha='center', va='bottom', fontsize=11)

plt.tight_layout()
plt.show()

### **1-5. ?덊듃留듭쑝濡?sentiment? emotion 愿怨??쒓컖??*

In [ ]:
pivot_table = pd.pivot_table(df, index='sentiment', columns='emotion', aggfunc='size', fill_value=0)
plt.figure(figsize=(8, 6))
sns.heatmap(pivot_table, annot=True, cmap='Blues', fmt='g')

plt.title('Relationship')
plt.xlabel('Emotion')
plt.ylabel('Sentiment')

plt.show()

### **1-6. 留됰?洹몃옒?꾨줈 datatime怨?sentiment 愿怨??쒓컖??*

In [ ]:
df['Datetime'] = pd.to_datetime(df['Datetime'])
df.set_index('Datetime', inplace=True)

# ?붾퀎 媛먯젙怨?sentiment 愿怨?吏묎퀎
sentiment_counts = df.groupby(df.index.to_period('M'))['sentiment'].value_counts().unstack().fillna(0)

# datetime ?뺤떇?쇰줈 ?몃뜳??蹂??nsentiment_counts.index = sentiment_counts.index.to_timestamp()

# 洹몃옒??洹몃━湲?nplt.figure(figsize=(10, 6))

plt.plot(sentiment_counts.index, sentiment_counts['neutral'], label='Neutral', color='green')
plt.plot(sentiment_counts.index, sentiment_counts['negative'], label='Negative', color='red')
plt.plot(sentiment_counts.index, sentiment_counts['positive'], label='Positive', color='blue')

plt.title('Monthly Sentiment Counts(2022)')
plt.xlabel('Month')
plt.ylabel('Count')

plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

### **1-7. 媛?媛먯젙蹂?text?먯꽌 媛??留롮씠 ?깆옣?섎뒗 ?⑥뼱 遺꾩꽍 - wordcloud**

In [ ]:
from collections import Counter
from wordcloud import WordCloud
import matplotlib.pyplot as plt
import nltk
from nltk.corpus import stopwords
import re

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

pattern = r'@\w+'
def preprocess_text(text):
    text = re.sub(pattern, '', text)  # '@'濡??쒖옉?섎뒗 ?⑥뼱 ?쒓굅
    return text

df['clean_text'] = df['Text'].apply(preprocess_text)

emotions = df['emotion'].unique()

plt.figure(figsize=(15, 10))

for i, emotion in enumerate(emotions):
    filtered_text = ' '.join(df[df['emotion'] == emotion]['clean_text'])
    filtered_text_out = ' '.join(word for word in filtered_text.split() if word.lower() not in stop_words)
    word_counts_out = Counter(filtered_text_out.split())
    top_words_out = dict(word_counts_out.most_common(20))

    # WordCloud ?앹꽦
    wordcloud = WordCloud(width=400, height=300, background_color='white').generate_from_frequencies(top_words_out)

    # subplot 異붽?
    plt.subplot(2, 4, i+1)  # 2??4?댁쓽 洹몃━?쒖뿉??i+1 踰덉㎏ subplot??洹몃┝
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.title(f'{emotion.capitalize()} - Top 20 words')
    plt.axis('off')

plt.tight_layout(pad=2.0)  # subplot 媛?媛꾧꺽 ?볤쾶 ?ㅼ젙
plt.show()

## **2. ?곗씠???꾩쿂由?*

In [ ]:
df

In [ ]:
df.reset_index(inplace=True)
df['Datetime'] = pd.to_datetime(df['Datetime']).dt.strftime('%y-%m-%d')
df_clean = df.drop(['Text', 'sentiment_score', 'emotion_score'], axis=1)

In [ ]:
df_clean

In [ ]:
# prompt: df_clean DataFrame ?ъ슜: clean_text瑜????⑹튇 all_text瑜?留뚮뱺 ?ㅼ쓬 媛?紐낆궗 ?⑥뼱??鍮덈룄?섎? 異붿텧?섎뒗 肄붾뱶瑜??묒꽦?댁＜?몄슂

import pandas as pd

# Combine all text into a single string
all_text = ' '.join(df_clean['clean_text'])

# Extract all nouns from the text
nouns = [word for word, pos in nltk.pos_tag(all_text.split()) if pos.startswith('N')]

# Count the frequency of each noun
noun_counts = pd.Series(nouns).value_counts()

# Print the noun counts
print(noun_counts)


## **3. Semantic Network Analysis(?뚮뱶?대씪?곕뱶 ?쒖쇅)**

In [ ]:
!pip show squarify
# Install dependency in the environment if needed
# Install dependency in the environment if needed


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

import squarify

import requests
from bs4 import BeautifulSoup

import nltk
nltk.download("punkt")

from nltk import word_tokenize, bigrams
from nltk.util import ngrams
from nltk import ConditionalFreqDist, FreqDist

from konlpy.tag import Kkma
from konlpy.tag import Okt

from wordcloud import WordCloud
from collections import Counter

import time
import datetime

import warnings
warnings.filterwarnings('ignore')

### **3-1. Treemap 洹몃━湲?*

In [ ]:
from collections import Counter
from konlpy.tag import Okt
from nltk.tag import pos_tag

nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

okt = Okt()

all_text = ' '.join(df_clean['clean_text'].tolist())
tokens = word_tokenize(all_text)
words = [word for word in tokens if len(word) > 1]
words_2 = [word for word in words if re.match(r'^[a-zA-Z]+$', word)]

def pos_tagging(words):
    tagged_words = pos_tag(words)
    return tagged_words

tagged_words = pos_tagging(words_2)
nouns = [word for word, pos in tagged_words if pos.startswith('NN')]


word_counts = Counter(nouns)
most_words = dict(word_counts.most_common(50))

cmap = matplotlib.cm.Set3
normalize = matplotlib.colors.Normalize(vmin=min(most_words.values()), vmax=max(most_words.values()))

colors = [cmap(normalize(value)) for value in most_words.values()]

plt.figure(figsize=(10, 10))
squarify.plot(label=most_words.keys(),
              sizes=most_words.values(),
              text_kwargs={'fontsize': 12},
              color=colors,
              ec='white',
              alpha=0.7)

plt.axis("off")
plt.show()

### **3-2. networkx 洹몃━湲?*

In [ ]:
import networkx as nx
nltk.download('wordnet')
import itertools
import operator

most_freq_words = [word for word, count in word_counts.most_common(10)]

G = nx.Graph()

# ?몃뱶 異붽?
G.add_nodes_from(most_freq_words)

# 臾몄옣?먯꽌 紐낆궗 ?⑥뼱?ㅼ쓣 異붿텧?섍퀬 ?ㅽ듃?뚰겕???ｌ? 異붽?
article_sentences = df_clean['clean_text']
repeat_num = 1

for sentence in article_sentences:
    word_tokens = nltk.word_tokenize(sentence)
    tokens_pos = nltk.pos_tag(word_tokens)

    NN_words = [word for word, pos in tokens_pos if 'NN' in pos]

    wlem = nltk.WordNetLemmatizer()
    lemmatized_words = [wlem.lemmatize(word) for word in NN_words]

    selected_words = [word for word in lemmatized_words if word in most_freq_words]

    selected_words = list(set(selected_words))  # 以묐났 ?쒓굅

    repeat_num += 1

    # ?몃뱶 媛꾩쓽 議고빀???ｌ?濡?異붽?
    for pair in itertools.combinations(selected_words, 2):
        if G.has_edge(pair[0], pair[1]):
            G[pair[0]][pair[1]]['weight'] += 1
        else:
            G.add_edge(pair[0], pair[1], weight=1)

# 媛以묒튂 湲곗??쇰줈 ?뺣젹
edge_items = nx.get_edge_attributes(G, 'weight')
sorted_edge_items = sorted(edge_items.items(), key=operator.itemgetter(1), reverse=True)

color_map = []
for node in G:
    if G.degree(node) >= 2:   # ?덉떆濡?degree媛 2 ?댁긽??寃쎌슦 'pink' ?됱긽
        color_map.append('pink')
    else:
        color_map.append('beige')

# circular layout?쇰줈 ?쒓컖??nplt.figure(figsize=(6, 6))
pos = nx.circular_layout(G, scale=0.2)

nx.draw_networkx(G, pos, node_color=color_map, edge_color='grey', with_labels=True, node_size=3000, font_size=10)

plt.axis('off')
plt.title('Network Graph with Node Color by Degree')
plt.show()

In [ ]:
def get_node_size(node_values):
    node_sizes = np.array(list(node_values))
    node_sizes = 1000 * (node_sizes - min(node_sizes)) / (max(node_sizes) - min(node_sizes)) + 300
    return node_sizes

# ?ㅼ뼇??以묒떖??吏??怨꾩궛
dc = nx.degree_centrality(G)  # Degree Centrality
cc = nx.closeness_centrality(G)  # Closeness Centrality
bc = nx.betweenness_centrality(G)  # Betweenness Centrality
ec = nx.eigenvector_centrality(G, weight='weight')  # Eigenvector Centrality
pr = nx.pagerank(G)   # PageRank

def get_node_size(node_values):
    node_sizes = np.array(list(node_values.values()))
    node_sizes = 1000 * (node_sizes - min(node_sizes)) / (max(node_sizes) - min(node_sizes)) + 300
    return node_sizes

# ?몃뱶 ?ш린 ?ㅼ젙
node_sizes = get_node_size(dc)  # ?덉떆濡?Degree Centrality瑜?湲곗??쇰줈 ?ㅼ젙

# ?ㅽ듃?뚰겕 ?쒓컖??nplt.figure(figsize=(8, 8))
pos = nx.spring_layout(G, seed=42)  # ?덉씠?꾩썐 ?ㅼ젙
nx.draw_networkx_nodes(G, pos, node_color='skyblue', node_size=node_sizes)
nx.draw_networkx_edges(G, pos, width=1.0, alpha=0.5, edge_color='grey')
nx.draw_networkx_labels(G, pos, font_size=12, font_color='black', font_family='sans-serif')
plt.title('Network Visualization', fontsize=15)
plt.axis('off')
plt.show()

In [ ]:
print("Degree Centrality:")
for node, centrality in dc.items():
    print(f"{node}: {centrality}")

print("\nCloseness Centrality:")
for node, centrality in cc.items():
    print(f"{node}: {centrality}")

print("\nBetweenness Centrality:")
for node, centrality in bc.items():
    print(f"{node}: {centrality}")

print("\nEigenvector Centrality:")
for node, centrality in ec.items():
    print(f"{node}: {centrality}")

print("\nPageRank:")
for node, centrality in pr.items():
    print(f"{node}: {centrality}")

## **4. Topic Modeling 遺꾩꽍**

In [ ]:
df = pd.read_csv("path/to/external/data")

In [ ]:
df = df.sample(n=1000, random_state=42)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

# Preprocess the text data
vectorizer = CountVectorizer(max_features=1000, stop_words='english')
X = vectorizer.fit_transform(df['Text'])

# Apply LDA
lda = LatentDirichletAllocation(n_components=5, random_state=42)
topics = lda.fit_transform(X)

# Print topics and visualize
print("Top words for each topic:")
feature_names = vectorizer.get_feature_names_out()
for topic_idx, topic in enumerate(lda.components_):
    top_words = [feature_names[i] for i in topic.argsort()[:-10 - 1:-1]]
    print(f"Topic #{topic_idx+1}: {' '.join(top_words)}")

In [ ]:
df_Sadness = df[df['emotion'] == 'sadness']
df_Joy = df[df['emotion'] == 'joy']
df_Anticipation = df[df['emotion'] == 'anticipation']
df_Anger = df[df['emotion'] == 'anger']
df_Surprise = df[df['emotion'] == 'surprise']
df_Fear = df[df['emotion'] == 'fear']
df_Optimism = df[df['emotion'] == 'optimism']
df_Disgust = df[df['emotion'] == 'disgust']

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

# CountVectorizer 諛?LDA ?곸슜 ?⑥닔 ?뺤쓽
def apply_lda(df, emotion_label):
    vectorizer = CountVectorizer(max_features=2000)  # max_features 媛믪쓣 ?섎┝
    X = vectorizer.fit_transform(df['Text'])

    lda = LatentDirichletAllocation(n_components=5, random_state=42)
    topics = lda.fit_transform(X)

    print(f"Top words for each topic in {emotion_label}:")
    feature_names = vectorizer.get_feature_names_out()
    for topic_idx, topic in enumerate(lda.components_):
        top_words = [feature_names[i] for i in topic.argsort()[:-10 - 1:-1]]
        print(f"Topic #{topic_idx+1}: {' '.join(top_words)}")
    print('----------------------')

In [ ]:
apply_lda(df_Sadness, 'Sadness')
apply_lda(df_Joy, 'Joy')
apply_lda(df_Anticipation, 'Anticipation')
apply_lda(df_Anger, 'Anger')
apply_lda(df_Surprise, 'Surprise')
apply_lda(df_Fear, 'Fear')
apply_lda(df_Optimism, 'Optimism')
apply_lda(df_Disgust, 'Disgust')

In [ ]:
# CountVectorizer 諛?LDA ?곸슜 ?⑥닔 ?뺤쓽
def apply_lda(df, emotion_label, n_topics, max_iter):
    vectorizer = CountVectorizer(max_features=2000)
    X = vectorizer.fit_transform(df['Text'])

    lda = LatentDirichletAllocation(n_components=n_topics, max_iter=max_iter, random_state=42)
    lda.fit(X)

    print(f"Top words for each topic in {emotion_label}:")
    feature_names = vectorizer.get_feature_names_out()
    for topic_idx, topic in enumerate(lda.components_):
        top_words = [feature_names[i] for i in topic.argsort()[:-10 - 1:-1]]
        print(f"Topic #{topic_idx + 1}: {' '.join(top_words)}")
    print('----------------------')

    return lda

# 媛먯젙蹂?LDA 紐⑤뜽 ?곸슜 諛?寃곌낵 異쒕젰
lda_anticipation = apply_lda(df[df['emotion'] == 'anticipation'], 'Anticipation', 10, 500)
lda_joy = apply_lda(df[df['emotion'] == 'joy'], 'Joy', 10, 500)
lda_anger = apply_lda(df[df['emotion'] == 'anger'], 'Anger', 10, 500)
lda_disgust = apply_lda(df[df['emotion'] == 'disgust'], 'Disgust', 10, 500)


## **5. Text classifiaction - BERT 紐⑤뜽 ?ъ슜**

In [ ]:
import pandas as pd
import torch
import transformers
from transformers import BertTokenizer, BertModel, AdamW, get_linear_schedule_with_warmup
from torch.utils.data import DataLoader, Dataset
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
from collections import defaultdict

In [ ]:
df_clf = df[['Text', 'emotion']][0:2000]
df_clf.columns = ['text', 'labels']
class_names = df_clf['labels'].unique()
print(class_names)

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
MAX_LEN = 128

In [ ]:
label_map = {label: idx for idx, label in enumerate(class_names)}
def map_label(label):
    return label_map[label]

In [ ]:
class GPReviewDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = map_label(self.labels[idx])

        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            truncation=True,
            return_token_type_ids=False,
            pad_to_max_length=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        return {
            'text': text,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'label': torch.tensor(label, dtype=torch.long)
        }

In [ ]:
train_dataset = GPReviewDataset(
    texts=df_clf['text'].values,
    labels=df_clf['labels'].values,
    tokenizer=tokenizer,
    max_len=MAX_LEN
)

In [ ]:
BATCH_SIZE = 16
train_data_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

In [ ]:
class BERTSentimentClassifier(nn.Module):
    def __init__(self, bert_model, num_classes):
        super(BERTSentimentClassifier, self).__init__()
        self.bert = bert_model
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.pooler_output
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)
        return logits

In [ ]:
import torch.optim as optim
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = len(class_names)
bert_model = BertModel.from_pretrained('bert-base-uncased', return_dict=True)
model = BERTSentimentClassifier(bert_model, num_classes)
model.to(device)

loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=2e-5)

In [ ]:
def train_epoch(model, data_loader, loss_fn, optimizer, device, scheduler, n_examples):
    model.train()
    losses = []
    correct_predictions = 0

    for d in data_loader:
        input_ids = d["input_ids"].to(device)
        attention_mask = d["attention_mask"].to(device)
        labels = d["label"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        _, preds = torch.max(outputs, dim=1)
        loss = loss_fn(outputs, labels)

        correct_predictions += torch.sum(preds == labels)
        losses.append(loss.item())

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

    return correct_predictions.double() / n_examples, np.mean(losses)

def eval_model(model, data_loader, loss_fn, device, n_examples):
    model.eval()
    losses = []
    correct_predictions = 0
    predictions = []
    real_values = []

    with torch.no_grad():
        for d in data_loader:
            input_ids = d["input_ids"].to(device)
            attention_mask = d["attention_mask"].to(device)
            labels = d["label"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            _, preds = torch.max(outputs, dim=1)
            loss = loss_fn(outputs, labels)

            correct_predictions += torch.sum(preds == labels)
            losses.append(loss.item())

            predictions.extend(preds)
            real_values.extend(labels)

    predictions = torch.stack(predictions).cpu()
    real_values = torch.stack(real_values).cpu()
    return correct_predictions.double() / n_examples, np.mean(losses), predictions, real_values


In [ ]:
EPOCHS = 10
total_steps = len(train_data_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

In [ ]:
for epoch in range(EPOCHS):
    print(f'Epoch {epoch + 1}/{EPOCHS}')
    print('-' * 10)

    train_acc, train_loss = train_epoch(model, train_data_loader, loss_fn, optimizer, device, scheduler, len(train_dataset))
    print(f'Train loss: {train_loss:.4f}, accuracy: {train_acc:.4f}')

In [ ]:
df_clf_test = df[['Text', 'emotion']][2001:2500]
df_clf_test.columns = ['text', 'labels']
test_dataset = GPReviewDataset(
    texts=df_clf_test['text'].values,
    labels=df_clf_test['labels'].values,
    tokenizer=tokenizer,
    max_len=MAX_LEN
)
test_data_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
test_acc, test_loss, predictions, real_values = eval_model(
    model,
    test_data_loader,
    loss_fn,
    device,
    len(test_dataset)
)

# ?됯? 寃곌낵 異쒕젰
print(f'Test loss: {test_loss:.4f}, accuracy: {test_acc:.4f}')

In [ ]:
y_pred = predictions.numpy()
y_true = real_values.numpy()

# ?쇰룞?됰젹 ?앹꽦
conf_mat = confusion_matrix(y_true, y_pred)

# ?쇰룞?됰젹 ?쒓컖??nplt.figure(figsize=(10, 8))
sns.heatmap(conf_mat, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
# y_pred? y_true?먯꽌 ?ъ슜???대옒???뺤씤
unique_classes_pred = np.unique(y_pred)
unique_classes_true = np.unique(y_true)
# ?ㅼ젣 ?ъ슜???대옒?ㅼ쓽 媛쒖닔
num_classes_pred = len(unique_classes_pred)
num_classes_true = len(unique_classes_true)

# target_names瑜??ㅼ젣 ?ъ슜???대옒???대쫫?쇰줈 ?쒗븳
actual_class_names = [class_names[i] for i in unique_classes_true]
print(f"Actual class names: {actual_class_names}")

# ?ㅼ젣 ?ъ슜???대옒???몃뜳?ㅻ? labels濡??꾨떖
class_report = classification_report(y_true, y_pred, labels=unique_classes_true, target_names=class_names[unique_classes_true])

# 遺꾨쪟 蹂닿퀬??異쒕젰
print(class_report)
